# Train the MRI model on OASIS-1 (Colab or Kaggle)

Runs the pipeline in `ml/` on a GPU and produces `model_export.zip` for the app.

**Before you start**
1. Request access to OASIS-1 at https://sites.wustl.edu/oasisbrains/home/access/ and agree to the data use terms. The data is for non-commercial research and may not be redistributed.
2. Put this project on GitHub (or zip the folder) so the notebook can fetch the code.
3. Turn on a GPU: *Runtime > Change runtime type > T4 GPU* (Colab) or *Settings > Accelerator* (Kaggle).

This notebook has not been run against the real OASIS servers by its author. The download cell tells you what to do if the server refuses.

In [ ]:
import torch, platform
print("Python", platform.python_version(), "| torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - training will be very slow")

## 1. Get the code
Set `REPO_URL` to your repository. To use a zip instead, upload `alz-screen.zip` to the session and leave `REPO_URL` empty.

In [ ]:
import os, subprocess, zipfile, pathlib
REPO_URL = ""   # e.g. "https://github.com/<you>/alz-screen.git"
WORK = pathlib.Path("/content" if os.path.exists("/content") else "/kaggle/working")
PROJECT = WORK / "alz-screen"
if not PROJECT.exists():
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)], check=True)
    else:
        zipfile.ZipFile(WORK / "alz-screen.zip").extractall(WORK)
os.chdir(PROJECT)
print("working in", PROJECT)

In [ ]:
%pip install -q monai nibabel onnx onnxruntime openpyxl

## 2. Get the data

Downloads each disc, keeps only the atlas-registered masked images (a few MB per person) and deletes the archive, so disk use stays small.

If you see HTTP 401 or 403, your access has not been approved yet or the server wants a login. In that case download the discs in your browser, upload them to Google Drive, and set `LOCAL_DISCS` to that folder.

In [ ]:
import pathlib, subprocess
DATA = pathlib.Path("data/oasis1"); DATA.mkdir(parents=True, exist_ok=True)
LOCAL_DISCS = ""   # folder that already holds oasis_cross-sectional_disc*.tar.gz, if you downloaded them yourself
BASE = "https://download.nrg.wustl.edu/data"
for i in range(1, 13):
    done = DATA / f".disc{i}.done"
    if done.exists():
        continue
    name = f"oasis_cross-sectional_disc{i}.tar.gz"
    tar = pathlib.Path(LOCAL_DISCS) / name if LOCAL_DISCS else DATA / name
    if not LOCAL_DISCS:
        r = subprocess.run(["curl", "-fL", "--retry", "3", "-o", str(tar), f"{BASE}/{name}"])
        if r.returncode != 0:
            raise SystemExit(f"Could not download {name}. See the note above this cell.")
    subprocess.run(["tar", "-xzf", str(tar), "-C", str(DATA), "--wildcards", "*T88_111/*_t88_masked_gfc.*"], check=True)
    if not LOCAL_DISCS:
        tar.unlink()
    done.touch()
    print("disc", i, "ready")

In [ ]:
# Demographics and clinical ratings. If this link has moved, download the "CSV" file from
# https://sites.wustl.edu/oasisbrains/oasis-1/ and upload it as data/oasis1/oasis_cross-sectional.xlsx
CSV = DATA / "oasis_cross-sectional.xlsx"
if not CSV.exists():
    subprocess.run(["curl", "-fL", "-o", str(CSV),
                    "https://sites.wustl.edu/oasisbrains/files/2024/04/oasis_cross-sectional-5708aa0a98d82080.xlsx"], check=True)
from ml.oasis import build_cohort, describe
cohort = build_cohort(CSV, DATA)
print(describe(cohort))

**Check the cohort before training.** You should see about 198 people, close to half with dementia, similar ages in both groups, and no CDR values missing. If the number is far off, some discs did not extract.

## 3. Train
Five folds, up to 60 epochs each. Time on a T4 has not been measured; the loop prints progress every five epochs.

In [ ]:
!python -m ml.train --data-root data/oasis1 --csv {CSV} --out runs/oasis_v1

## 4. Sanity check: shuffled labels

Same pipeline, labels permuted. The AUC **must** come out near 0.5. If it does not, there is a leak and the numbers in step 3 cannot be trusted.

In [ ]:
!python -m ml.train --data-root data/oasis1 --csv {CSV} --out runs/sanity --cache-dir runs/oasis_v1/cache --shuffle-labels --epochs 20

## 5. Results

In [ ]:
from IPython.display import Markdown
import json, pathlib
from ml.report import render
Markdown(render(json.loads(pathlib.Path("runs/oasis_v1/metrics.json").read_text())))

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from sklearn.calibration import calibration_curve
oof = pd.read_csv("runs/oasis_v1/oof_predictions.csv")
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
fpr, tpr, _ = roc_curve(oof.label, oof.prob); ax[0].plot(fpr, tpr); ax[0].plot([0, 1], [0, 1], "--", color="gray")
ax[0].set(xlabel="1 - specificity", ylabel="sensitivity", title="Out-of-fold ROC")
frac, mean = calibration_curve(oof.label, oof.prob, n_bins=5, strategy="quantile"); ax[1].plot(mean, frac, "o-"); ax[1].plot([0, 1], [0, 1], "--", color="gray")
ax[1].set(xlabel="predicted probability", ylabel="observed fraction with dementia", title="Calibration")
plt.tight_layout(); plt.show()

## 6. Export for the app

Unzip `model_export.zip` into `models/current/` in the project on your computer.

OASIS terms forbid redistributing the data. Before publishing trained weights anywhere public, read the data use agreement or ask the OASIS team; publishing code and metrics is the safe default.

In [ ]:
!python -m ml.export --run runs/oasis_v1 --dest models/current
!python -m ml.report runs/oasis_v1 > models/current/RESULTS.md
!cd models/current && zip -q -r ../../model_export.zip . && cd ../.. && ls -la model_export.zip